In [2]:
import pandas as pd
# Going to have to update this to your own file path
df = pd.read_csv(r"C:\Users\bandz\OneDrive\Desktop\New folder\archive (1)\vgchartz-2024.csv")
df.head()

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,NaN
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,NaN
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17,NaN
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2018-01-14


In [3]:
# Clean the data removing duplicates and dropping unnecessary columns

# Drop unnecessary columns
df = df.drop(columns=["img", "last_update"], errors="ignore")

# Drop regional sales columns
df = df.drop(columns=["na_sales", "jp_sales", "pal_sales", "other_sales"], errors="ignore")

# Remove duplicates (safe subset)
subset_cols = [
    "title", "console", "genre", "publisher", "developer",
    "critic_score", "total_sales"
]

if "release_date" in df.columns:
    subset_cols.append("release_date")

df = df.drop_duplicates(subset=subset_cols)

# Handle missing values
df["critic_score"] = df["critic_score"].fillna(df["critic_score"].median())
df = df.dropna(subset=["total_sales", "genre", "publisher", "developer", "console"])

# Reset index
df = df.reset_index(drop=True)

df.head()

,title,console,genre,publisher,developer,critic_score,total_sales,release_date
0,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,2013-09-17
1,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,2014-11-18
2,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,2002-10-28
3,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,7.5,15.86,2013-09-17
4,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,2015-11-06


In [4]:
# Feature engineering
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df["game_age"] = 2025 - df["release_year"]

df = df.dropna(subset=["release_year"])
df = df.drop(columns=["release_date"], errors="ignore")

In [5]:
# Prepare data for modeling

df = df.drop(columns=["score_bucket"], errors="ignore")

df_model = df.drop(columns=["title", "publisher", "developer"])

df_model = pd.get_dummies(
    df_model,
    columns=["console", "genre"],
    drop_first=True
)

# Preview
df_model.head()

# Check shape and types
print(df_model.shape)
print(df_model.dtypes)

(18825, 61)
critic_score          float64
total_sales           float64
release_year          float64
game_age              float64
console_3DO              bool
                       ...   
genre_Shooter            bool
genre_Simulation         bool
genre_Sports             bool
genre_Strategy           bool
genre_Visual Novel       bool
Length: 61, dtype: object


In [6]:
# Convert categorical variables into numerical

# Drop title so the model does not just memorize games
df_model = df.drop(columns=["title"])
df = df.drop(columns=["release_year"])

# One-hot encode categorical columns
df_model = pd.get_dummies(
    df_model,
    columns=["console", "genre", "publisher", "developer"],
    drop_first=True
)

# Preview
df_model.head()

# Check the result
print(df_model.shape)
print(df_model.dtypes)

(18825, 3655)
critic_score                               float64
total_sales                                float64
release_year                               float64
game_age                                   float64
console_3DO                                   bool
                                            ...   
developer_thatgamecompany                     bool
developer_thumbfood Ltd.                      bool
developer_tri-Ace                             bool
developer_tri-Crescendo / Monolith Soft       bool
developer_zSlide                              bool
Length: 3655, dtype: object


In [7]:
# Split feature into predictors and target (x and y)
y = df_model["total_sales"]
X = df_model.drop(columns=["total_sales"])

print(X.shape)
print(y.shape)

(18825, 3654)
(18825,)


In [8]:
# Split data into an 80/20 train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(15060, 3654) (3765, 3654)
(15060,) (3765,)


In [9]:
# Create and fit a random forest to train data
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=700,
    bootstrap=True,
    max_features="sqrt",
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,700
,criterion,'squared_error'
,max_depth,30
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
# Predict
y_pred = rf.predict(X_test)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

MAE: 0.29498029853734176
RMSE: 0.663262160448891
R^2: 0.35032004799335903


In [12]:
# Check feature importance
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

critic_score                0.202000
release_year                0.084909
game_age                    0.081442
developer_Infinity Ward     0.036893
developer_Treyarch          0.036022
console_PS4                 0.033855
developer_Rockstar North    0.026087
console_PC                  0.024367
genre_Shooter               0.023634
publisher_Rockstar Games    0.022169
dtype: float64

In [13]:
# Check for overfitting
y_train_pred = rf.predict(X_train)

from sklearn.metrics import r2_score
print("Train R^2:", r2_score(y_train, y_train_pred))
print("Test R^2:", r2_score(y_test, y_pred))

Train R^2: 0.5876561480365705
Test R^2: 0.35032004799335903
